In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mga yugto ng Transpiler

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.4.0
```
</details>
Inilalarawan ng pahinang ito ang mga yugto ng prebuilt transpilation pipeline sa Qiskit SDK. May anim na yugto:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

Ang function na [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) ay gumagawa ng preset na [staged pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) na binubuo ng mga yugto na ito. Ang mga tiyak na pass na bumubuo sa bawat yugto ay nakasalalay sa mga argumento na ipinasa sa `generate_preset_pass_manager`. Ang `optimization_level` ay isang positional argument na dapat tukuyin; ito ay isang integer na maaaring 0, 1, 2, o 3. Ang mas mataas na halaga ay nagpapahiwatig ng mas mabigat ngunit mas mahal na optimization (tingnan ang [Mga default na setting at opsyon sa configuration ng transpilation](defaults-and-configuration-options)).

Ang inirerekomendang paraan para i-transpile ang isang Circuit ay ang lumikha ng preset staged pass manager at pagkatapos ay patakbuhin ang pass manager na iyon sa Circuit, gaya ng inilarawan sa [Mag-transpile gamit ang mga pass manager](transpile-with-pass-managers). Gayunpaman, may mas simpleng ngunit hindi gaanong nako-customize na alternatibo — ang paggamit ng function na [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Tinatanggap ng function na ito ang Circuit nang direkta bilang argumento. Tulad ng sa `generate_preset_pass_manager`, ang mga tiyak na transpiler pass na ginagamit ay nakasalalay sa mga argumento, tulad ng `optimization_level`, na ipinasa sa `transpile`. Sa katunayan, sa loob ng `transpile` function, tinatawagan nito ang `generate_preset_pass_manager` para lumikha ng preset staged pass manager at pinapatakbo ito sa Circuit.
## Init stage
Ang unang yugto na ito ay halos walang ginagawa bilang default at pangunahing kapaki-pakinabang kung gusto mong isama ang iyong sariling mga paunang optimization. Dahil karamihan sa mga layout at routing algorithm ay idinisenyo lamang para magtrabaho sa single- at two-qubit gate, ginagamit din ang yugtong ito para isalin ang anumang gate na gumagana sa higit sa dalawang qubit sa mga gate na gumagana lamang sa isa o dalawang qubit.

Para sa karagdagang impormasyon tungkol sa pagpapatupad ng iyong sariling mga paunang optimization para sa yugtong ito, tingnan ang seksyon sa mga plugin at pag-customize ng mga pass manager.
## Layout stage
Ang susunod na yugto ay kinabibilangan ng layout o connectivity ng Backend na padadalhang Circuit.  Sa pangkalahatan, ang mga quantum circuit ay mga abstract na entity na ang mga qubit ay "virtual" o "logical" na representasyon ng mga tunay na qubit na ginagamit sa mga computation.  Para maisagawa ang isang pagkakasunud-sunod ng mga Gate, kinakailangan ang isang one-to-one na mapping mula sa "virtual" na mga qubit patungo sa "physical" na mga qubit sa isang tunay na quantum device.  Ang mapping na ito ay naka-imbak bilang isang `Layout` na object at bahagi ng mga constraint na tinukoy sa loob ng [instruction set architecture (ISA)](./transpile#instruction-set-architecture) ng isang Backend.

![Inilalarawan ng larawang ito ang mga qubit na nima-map mula sa wire representation patungo sa isang diagram na kumakatawan sa kung paano konektado ang mga qubit sa QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Qubit mapping")

Napakahalaga ng pagpili ng mapping para mabawasan ang bilang ng mga SWAP operation na kailangan para ma-map ang input Circuit sa device topology at matiyak na ang mga pinaka-well-calibrated na qubit ang gagamitin.  Dahil sa kahalagahan ng yugtong ito, sinusubukan ng mga preset pass manager ang ilang iba't ibang paraan para mahanap ang pinakamainam na layout.  Karaniwang nagsasangkot ito ng dalawang hakbang: una, subukang mahanap ang isang "perpekto" na layout (isang layout na hindi nangangailangan ng anumang SWAP operation), at pagkatapos, isang heuristic pass na nagsisikap na mahanap ang pinakamainam na layout na gagamitin kung hindi mahanap ang perpektong layout.  May dalawang `Passes` na karaniwang ginagamit para sa unang hakbang na ito:

- `TrivialLayout`: Simpleng nagma-map ng bawat virtual qubit sa parehong numbered na physical qubit sa device (hal., [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]).  Ito ay makasaysayang gawi na ginagamit lamang sa `optimzation_level=1` para subukang mahanap ang perpektong layout.  Kung mabigo ito, susunod na susubukan ang `VF2Layout`.
- `VF2Layout`: Ito ay isang `AnalysisPass` na pumipili ng ideal na layout sa pamamagitan ng pagtrato sa yugtong ito bilang isang subgraph isomorphism problem, na nalulutas ng VF2++ algorithm.  Kung mahiit sa isang layout ang mahanap, nagpapatakbo ng scoring heuristic para piliin ang mapping na may pinakamababang average error.

Pagkatapos para sa heuristic stage, dalawang pass ang ginagamit bilang default:

- `DenseLayout`: Hinahanap ang sub-graph ng device na may pinakamataas na connectivity at may parehong bilang ng mga qubit gaya ng Circuit  (ginagamit para sa optimization level 1 kung may mga control flow operation (tulad ng IfElseOp) na naroroon sa Circuit).
- `SabreLayout`: Pinipili ng pass na ito ang isang layout sa pamamagitan ng pagsisimula mula sa isang paunang random na layout at paulit-ulit na pagpapatakbo ng `SabreSwap` algorithm.  Ginagamit lamang ang pass na ito sa mga optimization level 1, 2, at 3 kung hindi mahanap ang perpektong layout sa pamamagitan ng `VF2Layout` pass.  Para sa karagdagang detalye tungkol sa algorithm na ito, tingnan ang papel na [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)
## Routing stage
Para mapatupad ang isang two-qubit gate sa pagitan ng mga qubit na hindi direktang konektado sa isang quantum device, kailangang maglagay ng isa o higit pang SWAP gate sa Circuit para maililipat ang mga qubit state hanggang sa maging magkatabing muli sila sa device gate map.  Bawat SWAP gate ay kumakatawan sa isang mahal at maingay na operasyon na isasagawa.  Kaya naman, ang paghahanap ng pinakamaliit na bilang ng mga SWAP gate na kailangan para ma-map ang isang Circuit sa isang ibinigay na device ay isang mahalagang hakbang sa proseso ng transpilation.  Para sa kahusayan, ang yugtong ito ay karaniwang kinakalkula kasabay ng Layout stage bilang default, ngunit lohikal silang naiiba sa isa't isa.  Ang *Layout* stage ay pumipili ng mga hardware qubit na gagamitin, habang ang *Routing* stage ay naglalagay ng angkop na dami ng SWAP gate para maisagawa ang mga Circuit gamit ang napiling layout.

Gayunpaman, mahirap mahanap ang optimal na SWAP mapping.  Sa katunayan, ito ay isang NP-hard problem, at samakatuwid ay labis na mahal para kalkulahin para sa lahat maliban sa pinakamaliit na mga quantum device at input circuit.  Para malampasan ito, gumagamit ang Qiskit ng stochastic heuristic algorithm na tinatawag na `SabreSwap` para makalkula ang isang maganda ngunit hindi necessarily optimal na SWAP mapping. Ang paggamit ng stochastic na pamamaraan ay nangangahulugang ang mga circuit na nabuo ay hindi garantisadong magiging pareho sa magkakasunod na pagpapatakbo.  Sa katunayan, ang paulit-ulit na pagpapatakbo ng parehong Circuit ay nagbubunga ng distribusyon ng mga circuit depth at bilang ng gate sa output.  Ito ang dahilan kung bakit maraming gumagamit ang pumipili na patakbuhin ang routing function (o ang buong `StagedPassManager`) nang maraming beses at piliin ang mga Circuit na may pinakamababang depth mula sa distribusyon ng mga output.

Halimbawa, tingnan natin ang isang 15-qubit GHZ circuit na pinatakbo nang 100 beses, gamit ang isang "masamang" (disconnected) na `initial_layout`.

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(15)


ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/62479697-cef1-4a89-ac2a-20051dd294f4-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ab89c4ea-06c4-4320-b493-feb691b3570d-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](/docs/guides/transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

Gaya ng makikita, kailangang magsagawa ang Circuit na ito ng isang two-qubit gate sa pagitan ng mga qubit 0 at 14, na napakalayo sa connectivity graph.  Ang pagpapatakbo ng Circuit na ito ay nangangailangan ng paglalagay ng mga SWAP gate para maisagawa ang lahat ng two-qubit gate gamit ang `SabreSwap` pass.

Tandaan din na ang `SabreSwap` algorithm ay naiiba mula sa mas malaking `SabreLayout` na pamamaraan sa nakaraang yugto.  Bilang default, pinapatakbo ng `SabreLayout` ang parehong layout at routing, at ibinalik ang na-transform na Circuit.  Ginagawa ito para sa ilang partikular na teknikal na dahilan na tinukoy sa [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout) ng pass.
## Translation stage
Kapag nagsusulat ng quantum circuit, malaya kang gumamit ng anumang quantum gate (unitary operation) na gusto mo, kasama ang koleksyon ng mga non-gate na operasyon tulad ng mga tagubilin sa pagsukat ng qubit o reset.  Gayunpaman, karamihang quantum device ay likas na sumusuporta lamang sa iilang quantum gate at non-gate na operasyon.  Ang mga native gate na ito ay bahagi ng kahulugan ng [ISA](./transpile#instruction-set-architecture) ng isang target at ang yugtong ito ng preset na `PassManagers` ay nagsasalin (o *unrolls*) ng mga gate na tinukoy sa isang Circuit sa mga native basis gate ng isang tinukoy na Backend.  Ito ay isang mahalagang hakbang, dahil pinapayagan nito ang Circuit na maisagawa ng Backend, ngunit karaniwang nagdudulot ng pagtaas sa depth at bilang ng mga Gate.

Dalawang espesyal na kaso ang partikular na mahalaga na i-highlight, at nakakatulong silang ilarawan kung ano ang ginagawa ng yugtong ito.

1. Kung ang isang SWAP gate ay hindi isang native gate sa target na Backend, nangangailangan ito ng tatlong CNOT gate:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2700405a-f559-45d3-99a9-5b4447621743-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

Bilang produkto ng tatlong CNOT gate, ang isang SWAP ay isang mahal na operasyon na isasagawa sa mga maingay na quantum device.  Gayunpaman, karaniwang kailangan ang ganitong mga operasyon para ma-embed ang isang Circuit sa limitadong gate connectivity ng maraming device.  Kaya naman, ang pagbabawas ng bilang ng mga SWAP gate sa isang Circuit ay isang pangunahing layunin sa proseso ng transpilation.

2. Ang Toffoli, o controlled-controlled-not gate (`ccx`), ay isang three-qubit gate.  Dahil ang aming basis gate set ay kinabibilangan lamang ng single- at two-qubit gate, ang operasyong ito ay kailangang i-decompose.  Gayunpaman, ito ay medyo mahal:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = GenericBackendV2(5)

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ae2d9390-5c26-46f3-9418-a684ba8a406a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

Para sa bawat Toffoli gate sa isang quantum circuit, maaaring magsagawa ang hardware ng hanggang anim na CNOT gate at ilang single-qubit gate.  Ipinapakita ng halimbawang ito na ang anumang algorithm na gumagamit ng maraming Toffoli gate ay magtatapos bilang isang Circuit na may malaking depth at samakatuwid ay makabuluhang maaapektuhan ng ingay.
## Optimization stage
Ang yugtong ito ay nakatuon sa pag-decompose ng mga quantum circuit sa basis gate set ng target na device, at kailangang labanan ang pagtaas ng depth mula sa mga layout at routing stage.  Sa kabutihang palad, maraming gawain para sa pag-optimize ng mga Circuit sa pamamagitan ng pagsasama o pag-aalis ng mga Gate.  Sa ilang kaso, napaka-epektibo ng mga pamamaraang ito kaya ang mga output circuit ay may mas mababang depth kaysa sa mga input, kahit na pagkatapos ng layout at routing sa hardware topology.  Sa ibang kaso, hindi masyadong magagawa, at maaaring mahirap isagawa ang computation sa mga maingay na device.  Ito ang yugto kung saan nagsisimulang mag-iba ang iba't ibang optimization level.

- Para sa `optimization_level=1`, inihahanda ng yugtong ito ang [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) at [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), na pinagsasama ang mga chain ng single-qubit gate at nagkakansela ng anumang magkasunod na CNOT gate.
- Para sa `optimization_level=2`, ginagamit ng yugtong ito ang [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass sa halip na `CXCancellation`, na nag-aalis ng mga redundant na gate sa pamamagitan ng pagsasamantala sa mga commutation relation.
- Para sa `optimization_level=3`, inihahanda ng yugtong ito ang mga sumusunod na pass:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

Bukod dito, nagsasagawa rin ang yugtong ito ng ilang panghuling pagsusuri para matiyak na ang lahat ng tagubilin sa Circuit ay binubuo ng mga basis gate na available sa target na Backend.

Ipinapakita ng halimbawa sa ibaba gamit ang isang GHZ state ang mga epekto ng iba't ibang setting ng optimization level sa circuit depth at bilang ng gate.

> **Note:** Nag-iiba-iba ang output ng transpilation dahil sa stochastic SWAP mapper. Kaya naman, ang mga numerong nasa ibaba ay malamang na magbabago sa bawat beses na patakbuhin mo ang code.

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state bago ang transpilation")

Ang sumusunod na code ay gumagawa ng isang 15-qubit GHZ state at inihambing ang mga `optimization_levels` ng transpilation sa mga tuntunin ng resultang circuit depth, bilang ng gate, at bilang ng multi-qubit gate.